# DepthWizard v4 — Kaggle 2× T4

**Accelerator: `GPU T4 x2`. Internet: on. `HF_TOKEN` added under Add-ons → Secrets.**

Start the real run with **Save Version → Save & Run All (Commit)**. The headless
commit is what gets the full ~12 h and persists `/kaggle/working` as the
notebook's output; an interactive session idles out long before the run ends.

Data is prepared in a *separate CPU notebook* (`bash run_kaggle.sh prepare`) and
attached here as an input dataset — see README §5.4.


In [ ]:
# 1. Code + deps
!git clone --depth 1 https://github.com/AbhayKale332/DepthWizard.git /kaggle/working/DepthWizard 2>/dev/null || true
%cd /kaggle/working/DepthWizard/FineTunning/V4_Kaggle
!pip install -q -r requirements.txt
!bash run_kaggle.sh check


In [ ]:
# 2. HF token (DINOv3-SAT and GAMUS are both gated)
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("token set:", bool(os.environ["HF_TOKEN"]))


In [ ]:
# 3. Point a single data_root at whatever was attached.
#    Symlinks only — nothing is copied, so this costs no quota and no time.
!bash run_kaggle.sh link


In [ ]:
# 4. Train.  Two processes, one per T4.
#    Session 1:  as-is.
#    Session 2:  append --resume /kaggle/input/<session-1-output>/outputs/v4/last_full.pt
#    Both sessions keep the SAME --max_minutes; only --session_minutes caps this one.
!bash run_kaggle.sh train


## If the session dies

`last_full.pt` in the committed output carries the optimiser, scaler, EMA,
teacher, RNG and elapsed minutes. Attach that version as an input to a fresh
notebook and add `--resume /kaggle/input/<name>/outputs/v4/last_full.pt` to
cell 4. `run.log` should show the LR continuing *down* the cosine, not re-warming.

If training finished but the exports did not, run `bash run_kaggle.sh finalize`
in a short single-GPU notebook against the committed `best.pt`.
